# Track B: Recommender System - Spotify Playlist Continuation

Datasets: Spotify Tracks Dataset (Kaggle) + Playlist2vec (Zenodo, MPD / RecSys Challenge 2018).
Framed as **playlist continuation**, not per-user rating prediction: public Spotify data has
no persistent user id, so playlist membership is used as the implicit preference signal.
Full rationale, citations and ethics discussion in Essay.


## 1. Environment Setup & Data Loading

In [ ]:
from pathlib import Path
from contextlib import closing
import sqlite3
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno
from dotenv import load_dotenv

import kaggle
from zenodo_get import download
from mysql_to_sqlite import load_mysql_dump_to_sqlite

from scipy.sparse import coo_matrix
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error, ndcg_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from matplotlib.ticker import FormatStrFormatter

import shap
import matplotlib.pyplot as plt
import pickle

import lime.lime_tabular

RANDOM_SEED = 3
np.random.seed(RANDOM_SEED)

load_dotenv()

## 2. Kaggle Spotify Tracks Dataset (audio features)

In [ ]:
# Load Kaggle Spotify Selection Dataset
datasets_path = Path("dataset")
csv_file = datasets_path / "dataset.csv"

if not csv_file.exists():
    print("Downloading Spotify Selection Dataset from Kaggle...")
    kaggle.api.dataset_download_files(
        "maharshipandya/-spotify-tracks-dataset", path=datasets_path, unzip=True
    )

df_tracks = pd.read_csv(csv_file)

# Drop index column if present
if "Unnamed: 0" in df_tracks.columns:
    df_tracks = df_tracks.drop(columns=["Unnamed: 0"])

print(
    f"Dataset successfully loaded. Total rows: {len(df_tracks):,}, Total columns: {len(df_tracks.columns)}"
)

In [ ]:
df_tracks.info()  # Initial Inspection
missingno.matrix(df_tracks)
df_tracks.describe()

# look for duplicates
all_tracks = len(df_tracks)
unique_tracks = len(
    df_tracks[["artists", "track_name", "album_name", "tempo"]].drop_duplicates()
)
unique_tracks_w_genre = len(
    df_tracks[
        ["artists", "track_name", "album_name", "tempo", "track_genre"]
    ].drop_duplicates()
)
print(
    f"Total tracks: {all_tracks} - Unique tracks: {unique_tracks} - Unique tracks by genre: {unique_tracks_w_genre}"
)

### Notes

No nulls. Row count exceeds unique tracks because the source has one row per
`(track_id, genre)` pair, so genres are aggregated into a list per track before deduping.


In [ ]:
df_tracks.head()  # Preview first 5 rows

In [ ]:
# Aggregate multi-genre rows into a single genres_list per track, then deduplicate
df_genres = df_tracks.groupby("track_id")["track_genre"].apply(list).reset_index()
df_genres.rename(columns={"track_genre": "genres_list"}, inplace=True)

df_unique = df_tracks.drop_duplicates(subset=["track_id"]).copy()
df_clean = df_unique.merge(df_genres, on="track_id", how="left")
df_clean = df_clean.drop(columns=["track_genre"])
display(df_clean)

In [ ]:
# Feature engineering over strings
df_engineered = df_clean.copy()
df_engineered["artists_list"] = (
    df_clean["artists"].fillna("").apply(lambda x: x.split(";"))
)
df_engineered = df_engineered.drop(columns=["artists"], errors="ignore")

In [ ]:
# tempo == 0 / time_signature == 0 indicate silence or a corrupt measurement, not a
# genuine value (key == 0 is a valid value, "C", so it is left untouched).
df_engineered_1 = df_engineered.copy()
corrupted_tracks = df_engineered[df_engineered_1["tempo"] == 0]
print(f"Tracks with tempo = 0: {len(corrupted_tracks)}")

if len(corrupted_tracks) > 0:
    print("Tempo = 0 tracks sample:")
    display(
        corrupted_tracks[
            ["track_name", "genres_list", "tempo", "danceability", "energy"]
        ].head()
    )

df_engineered_1 = df_engineered_1[df_engineered_1["tempo"] > 0]
print(f"Final dataset contains: {len(df_engineered_1):,} rows.")

In [ ]:
# Impute time_signature == 0 with the mode (categorical value, not meaningfully zero)
df_engineered_2 = df_engineered_1.copy()
mode_time_signature = df_engineered_2["time_signature"].mode()[0]
print(f"time_signature mode is: {mode_time_signature}")
num_zeros_ts = len(df_engineered_2[df_engineered_2["time_signature"] == 0])
print(f"Replacing time_signature = 0 with the mode for {num_zeros_ts} tracks...")
df_engineered_2["time_signature"] = df_engineered_2["time_signature"].replace(
    0, mode_time_signature
)
print("Verify imputation:")
print(df_engineered_2["time_signature"].value_counts())

In [ ]:
display(df_engineered_2)

## 3. Playlist2vec Dataset (playlist-track membership)

This dataset is not filtered by user (it has no persistent user id), only by playlist, so
each `playlist` is treated as the unit of implicit preference rather than an individual.


In [ ]:
# Zenodo dataset: Playlist2vec - Spotify Million Playlist Dataset (10.7 GB MySQL dump)
if not (datasets_path / "spotifydbdumpshare.sql").exists():
    download("10.5281/zenodo.5002584", output_dir=datasets_path)

### Loading the MySQL dump into SQLite

The dump is ~10.7 GB; `mysql_to_sqlite.py` streams it in without loading it fully into
memory. This is a one-off cost, reused via `spotify.db` on later runs.


In [ ]:
sql_files = sorted(datasets_path.glob("*.sql"))
schema_path = next(p for p in sql_files if "schema" in p.name.lower())
data_path = next(p for p in sql_files if p != schema_path)

db_path = datasets_path / "spotify.db"
if not db_path.exists():
    load_mysql_dump_to_sqlite(schema_path, data_path, db_path)

In [ ]:
# Sanity-check the assumption that track.id already equals the id embedded in track.uri
with closing(sqlite3.connect(db_path)) as conn:
    id_check = pd.read_sql_query(
        "SELECT id, uri, substr(uri, 15) AS uri_track_id, id = substr(uri, 15) AS ids_match FROM track LIMIT 5",
        conn,
    )
display(id_check)

### Sampling strategy

The full dump has ~1M playlists / ~3M tracks: too large to process end-to-end in the time
available. A random sample of playlists is drawn **before** the join, and only tracks that
already have Kaggle audio features are kept (a track without features can't be modelled
regardless of how many playlists it appears in).


In [ ]:
# random sampling was discarded for breaking the collaborative signal,
# and a 5-core filtering was applied to ensure that each playlist and song retained at least 5 interactions,
# complying with technical literature to mitigate noise in power-law distributions.

with closing(sqlite3.connect(db_path)) as conn:
    print("Mapping the acoustic catalog...")
    df_sql_tracks = pd.read_sql_query(
        "SELECT id AS dump_track_id, substr(uri, 15) AS track_id FROM track", conn
    )

    # keep IDs that exist in Kaggle cleaning
    valid_tracks = df_sql_tracks[
        df_sql_tracks["track_id"].isin(df_engineered_2["track_id"])
    ]
    valid_dump_ids = set(valid_tracks["dump_track_id"])

    print("Loading 7.2 million interactions into memory...")
    df_track_playlist = pd.read_sql_query(
        "SELECT playlist_id, track_id AS dump_track_id FROM track_playlist1", conn
    )

# first filter: discard songs that do not have audio features
df_track_playlist = df_track_playlist[
    df_track_playlist["dump_track_id"].isin(valid_dump_ids)
]

# k-core iterative filtering (The academic standard)
# A 5-core filtering strategy was employed to remove inactive users and infrequently interacted items,
# ensuring that each user had at least five interactions,
# following common preprocessing practices in sequential recommendation research
k = 5
print(f"Applying filter {k}-core...")
start_time = time.time()

while True:
    p_counts = df_track_playlist["playlist_id"].value_counts()
    t_counts = df_track_playlist["dump_track_id"].value_counts()

    valid_p = p_counts[p_counts >= k].index
    valid_t = t_counts[t_counts >= k].index

    if len(valid_p) == len(p_counts) and len(valid_t) == len(t_counts):
        break  # balance reached, no more filtering needed

    df_track_playlist = df_track_playlist[
        df_track_playlist["playlist_id"].isin(valid_p)
        & df_track_playlist["dump_track_id"].isin(valid_t)
    ]

print(f"Completed in {time.time() - start_time:.2f} seconds.")
print(f"Final interactions after {k}-core filtering: {len(df_track_playlist):,}")

# rebuild df_interactions for training
df_interactions = df_track_playlist.merge(
    valid_tracks, on="dump_track_id", how="inner"
).drop(columns=["dump_track_id"])

df_interactions = df_interactions.merge(
    df_engineered_2, on="track_id", how="inner", suffixes=("", "_track")
)

### Building the combined implicit-feedback table

Each row = one (playlist, track) interaction: the implicit positive label the models will
learn from. Restricted to the sampled playlists and to tracks retrieved from the dump.


In [ ]:
df_sql_tracks.columns

In [ ]:
df_engineered_2.columns

## 4. Sparsity Check

Most playlists contain a small fraction of all tracks, and most tracks appear in few
playlists. This affects both model choice (matrix factorisation assumes sparsity) and
validation strategy (cold-start playlists/tracks need explicit handling in the split).


In [ ]:
n_playlists = df_interactions["playlist_id"].nunique()
n_tracks = df_interactions["track_id"].nunique()
n_interactions = len(df_interactions)
density = n_interactions / (n_playlists * n_tracks)

print(f"Unique playlists: {n_playlists:,}")
print(f"Unique tracks:    {n_tracks:,}")
print(f"Interactions:     {n_interactions:,}")
print(f"Matrix density:   {density:.6f}  (sparsity: {1 - density:.6f})")

interactions_per_playlist = df_interactions.groupby("playlist_id").size()
interactions_per_track = df_interactions.groupby("track_id").size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(interactions_per_playlist, bins=50, ax=axes[0])
axes[0].set_title("Tracks per playlist")
sns.histplot(interactions_per_track, bins=50, ax=axes[1])
axes[1].set_title("Playlists per track")
plt.tight_layout()
plt.show()

print(f"Playlists with a single track: {(interactions_per_playlist == 1).sum():,}")
print(f"Tracks appearing in a single playlist: {(interactions_per_track == 1).sum():,}")

## 5. Ready for Modelling

To do next: task framing (NDCG@k / Recall@k), negative sampling, split by playlist,
classical ML baseline, DL model, explainability, bias/diagnostic analysis, ethics.
Details in companion document.


In [ ]:
# Map IDs to numeric indices for the matrix
playlist_map = {
    id: idx for idx, id in enumerate(df_interactions["playlist_id"].unique())
}
track_map = {id: idx for idx, id in enumerate(df_interactions["track_id"].unique())}

df_interactions["user_idx"] = df_interactions["playlist_id"].map(playlist_map)
df_interactions["item_idx"] = df_interactions["track_id"].map(track_map)
df_interactions["rating"] = 1.0  # implicit feedback: the interaction exists

# Train/Test Split (80/20)
train_df, test_df = train_test_split(df_interactions, test_size=0.2, random_state=3)

# build the Sparse Matrix only with Train
num_users = len(playlist_map)
num_items = len(track_map)

train_matrix = coo_matrix(
    (train_df["rating"], (train_df["user_idx"], train_df["item_idx"])),
    shape=(num_users, num_items),
).tocsr()

# Instantiate and Train SVD (Baseline)
n_components = 50  # key hyperparameter to justify
svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_SEED)

print("Training TruncatedSVD...")
# user_factors = U * Sigma ; item_factors = V^T
user_factors = svd.fit_transform(train_matrix)
item_factors = svd.components_

# Calculate RMSE over Test
test_users = test_df["user_idx"].values
test_items = test_df["item_idx"].values
y_true = test_df["rating"].values

# Vectorised prediction: dot product between user and item latent factors
y_pred = np.sum(user_factors[test_users] * item_factors[:, test_items].T, axis=1)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"SVD Baseline RMSE: {rmse:.4f}")

In [ ]:
# --- 1. CLASS DEFINITIONS (Must be initialized every time) ---
class SpotifyDataset(Dataset):
    def __init__(self, df, audio_cols):
        self.users = torch.tensor(df["user_idx"].values, dtype=torch.long)
        self.items = torch.tensor(df["item_idx"].values, dtype=torch.long)
        self.audio = torch.tensor(df[audio_cols].values, dtype=torch.float32)
        self.ratings = torch.tensor(df["rating"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.audio[idx], self.ratings[idx]


class TwoTowerModel(nn.Module):
    def __init__(self, num_users, num_items, num_audio_features, embed_dim=32):
        super().__init__()

        # User Tower (Playlist IDs)
        self.user_embedding = nn.Embedding(num_users, embed_dim)

        # Item Tower (Track IDs)
        self.item_embedding = nn.Embedding(num_items, embed_dim)

        # Audio Tower (BatchNorm added to prevent internal covariate shift)
        self.audio_layer = nn.Sequential(
            nn.Linear(num_audio_features, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, embed_dim),
        )

        # Fusion Layer (BatchNorm added to stabilize outputs before prediction)
        self.item_fusion = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.BatchNorm1d(embed_dim), nn.ReLU()
        )

    def forward(self, user_idx, item_idx, audio_features):
        u_emb = self.user_embedding(user_idx)
        i_emb = self.item_embedding(item_idx)

        # Process acoustic features
        a_emb = self.audio_layer(audio_features)

        # Concatenate track ID embeddings with acoustic embeddings
        i_combined = self.item_fusion(torch.cat([i_emb, a_emb], dim=1))

        # Calculate geometric similarity (dot product) and map to probabilities
        dot_product = (u_emb * i_combined).sum(dim=1)
        return torch.sigmoid(dot_product)


# --- 2. DYNAMIC EXECUTION & CACHING ---
CACHE_FILE = Path("cache/dl_cache.pkl")
MODEL_FILE = Path("models/two_tower_model.pth")

# Create directories if they don't exist
CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
MODEL_FILE.parent.mkdir(parents=True, exist_ok=True)

if CACHE_FILE.exists() and MODEL_FILE.exists():
    print("Loading cached data and trained model from disk...")
    with open(CACHE_FILE, "rb") as f:
        cache = pickle.load(f)

    playlist_map = cache["playlist_map"]
    track_map = cache["track_map"]
    train_dl_df = cache["train_dl_df"]  # Required for SHAP background
    test_dl_df = cache["test_dl_df"]
    audio_features = cache["audio_features"]

    num_users = len(playlist_map)
    num_items = len(track_map)

    model = TwoTowerModel(num_users, num_items, len(audio_features))
    model.load_state_dict(torch.load(MODEL_FILE, weights_only=True))
    model.eval()

    print("Model ready for inference and explainability (SHAP).")

else:
    print("Cache not found. Starting preprocessing, negative sampling, and training...")

    # 2.1 PREPARE DATA & SPLIT
    playlist_map = {
        id_val: idx
        for idx, id_val in enumerate(df_interactions["playlist_id"].unique())
    }
    track_map = {
        id_val: idx for idx, id_val in enumerate(df_interactions["track_id"].unique())
    }

    df_interactions["user_idx"] = df_interactions["playlist_id"].map(playlist_map)
    df_interactions["item_idx"] = df_interactions["track_id"].map(track_map)
    df_interactions["rating"] = 1.0

    train_df, test_df = train_test_split(
        df_interactions, test_size=0.2, random_state=RANDOM_SEED
    )

    # 2.2 NORMALIZE AUDIO FEATURES
    audio_features = [
        "danceability",
        "energy",
        "loudness",
        "speechiness",
        "acousticness",
        "instrumentalness",
        "liveness",
        "valence",
        "tempo",
    ]

    scaler = StandardScaler()
    scaler.fit(train_df[audio_features])  # fit ONLY with train

    train_df[audio_features] = scaler.transform(train_df[audio_features])
    test_df[audio_features] = scaler.transform(test_df[audio_features])
    df_engineered_2[audio_features] = scaler.transform(df_engineered_2[audio_features])

    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[audio_features] = scaler.transform(train_df[audio_features])
    test_df[audio_features] = scaler.transform(test_df[audio_features])

    positive_pairs = set(
        zip(df_interactions["playlist_id"], df_interactions["track_id"])
    )  # explicit filter for real possitive interactions

    # STATIC NEGATIVE SAMPLING
    def generate_negatives(df, all_users, all_items, df_features, n_tries=3):
        neg_users = np.random.choice(all_users, size=len(df) * n_tries)
        neg_items = np.random.choice(all_items, size=len(df) * n_tries)
        mask = [(u, i) not in positive_pairs for u, i in zip(neg_users, neg_items)]
        neg_users, neg_items = neg_users[mask][: len(df)], neg_items[mask][: len(df)]

        neg_df = pd.DataFrame(
            {"playlist_id": neg_users, "track_id": neg_items, "rating": 0.0}
        )
        neg_df = neg_df.merge(
            df_features[["track_id"] + audio_features], on="track_id", how="left"
        )
        neg_df["user_idx"] = neg_df["playlist_id"].map(playlist_map)
        neg_df["item_idx"] = neg_df["track_id"].map(track_map)
        return neg_df

    all_u = df_interactions["playlist_id"].unique()
    all_i = df_interactions["track_id"].unique()

    train_neg = generate_negatives(train_df, all_u, all_i, df_engineered_2)
    test_neg = generate_negatives(test_df, all_u, all_i, df_engineered_2)

    train_dl_df = (
        pd.concat([train_df, train_neg]).dropna().sample(frac=1).reset_index(drop=True)
    )
    test_dl_df = pd.concat([test_df, test_neg]).dropna().reset_index(drop=True)

    # 2.4 TRAINING LOOP
    train_dataset = SpotifyDataset(train_dl_df, audio_features)
    test_dataset = SpotifyDataset(test_dl_df, audio_features)

    train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

    num_users = len(playlist_map)
    num_items = len(track_map)
    model = TwoTowerModel(num_users, num_items, len(audio_features))

    criterion = nn.BCELoss()
    # Learning rate adjusted and L2 Regularization added
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-5)

    print("Starting Deep Learning Training with revised architecture...")
    epochs = 5
    epoch_losses = []
    for epoch in range(epochs):
        # generate new negatives for each epoch to introduce variability and prevent overfitting
        train_neg = generate_negatives(train_df, all_u, all_i, df_engineered_2)
        train_dl_df = (
            pd.concat([train_df, train_neg])
            .dropna()
            .sample(frac=1)
            .reset_index(drop=True)
        )
        train_dataset = SpotifyDataset(train_dl_df, audio_features)
        train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

        model.train()
        total_loss = 0
        for batch_users, batch_items, batch_audio, batch_ratings in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_users, batch_items, batch_audio)
            loss = criterion(predictions, batch_ratings)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        epoch_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f}")
        epoch_losses.append(epoch_loss)  # <-- AÑADIDO

    # 2.5 RMSE EVALUATION
    model.eval()
    all_preds = []
    all_trues = []

    with torch.no_grad():
        for batch_users, batch_items, batch_audio, batch_ratings in test_loader:
            preds = model(batch_users, batch_items, batch_audio)
            all_preds.extend(preds.numpy())
            all_trues.extend(batch_ratings.numpy())

    dl_rmse = np.sqrt(mean_squared_error(all_trues, all_preds))
    print(f"Deep Learning Two-Tower RMSE: {dl_rmse:.4f}")

    # 2.6 SAVE TO DISK
    print("Saving model and required test data to disk...")
    torch.save(model.state_dict(), MODEL_FILE)

    cache = {
        "playlist_map": playlist_map,
        "track_map": track_map,
        "train_dl_df": train_dl_df,
        "test_dl_df": test_dl_df,
        "audio_features": audio_features,
        "epoch_losses": epoch_losses,
    }
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(cache, f)

    print("Saved successfully. Future executions will bypass training.")

In [ ]:
# SELECT A SAMPLE TO EXPLAIN
sample_idx = 0
user_val = test_dl_df.iloc[sample_idx]["user_idx"]
item_val = test_dl_df.iloc[sample_idx]["item_idx"]
actual_rating = test_dl_df.iloc[sample_idx]["rating"]

print(f"Explaining prediction for User {user_val}, Item {item_val}")
print(f"Actual implicit rating: {actual_rating}")


# CREATE A WRAPPER FUNCTION FOR SHAP
def predict_audio_impact(audio_numpy):
    # Convert incoming SHAP permutations back to PyTorch tensors
    audio_tensor = torch.tensor(audio_numpy, dtype=torch.float32)
    batch_size = audio_tensor.shape[0]

    # Duplicate the fixed user and item IDs to match the batch size
    u_idx = torch.tensor([user_val] * batch_size, dtype=torch.long)
    i_idx = torch.tensor([item_val] * batch_size, dtype=torch.long)

    # Run through the model without calculating gradients
    with torch.no_grad():
        preds = model(u_idx, i_idx, audio_tensor)

    return preds.numpy()


# BACKGROUND DATASET
# SHAP needs a background distribution to know what "normal" audio features look like.
# Using k-means to summarize the training rows into 50 representative centroids to save RAM/Time (arbitrary choice of 50 clusters for efficiency)
background_data = shap.kmeans(train_dl_df[audio_features].values, 50)
explainer = shap.KernelExplainer(
    predict_audio_impact, background_data
)  # CALCULATE SHAP VALUES

# Extract the actual audio features for our chosen sample
test_sample = test_dl_df.iloc[sample_idx : sample_idx + 1][audio_features].values

# Compute the Shapley values
print("Calculating Shapley values (this might take a few seconds)...")
shap_values = explainer.shap_values(test_sample)

# Visualisation of the SHAP values to understand feature contributions
shap_exp = shap.Explanation(
    values=shap_values[0],
    base_values=explainer.expected_value,
    data=test_sample[0],
    feature_names=audio_features,
)

shap.waterfall_plot(shap_exp)
plt.show()

In [ ]:
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=train_dl_df[audio_features]
    .sample(50000, random_state=RANDOM_SEED)
    .values,
    feature_names=audio_features,
    mode="regression",  # Our model outputs a continuous probability (0 to 1)
)

# Explain the same song as SHAP (sample_idx = 0)
print("Calculating LIME for the same interaction...")
lime_exp = lime_explainer.explain_instance(
    data_row=test_sample[0], predict_fn=predict_audio_impact, num_features=5
)


fig = lime_exp.as_pyplot_figure()
ax = fig.gca()

ax.xaxis.set_major_formatter(
    FormatStrFormatter("%.4f")
)  # style adjustment to avoid overlapping
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# LEAVE-ONE-OUT (LOO) EVALUATION

# Toggle this flag to True ONLY if you need to recalculate the 80-minute evaluation
RUN_MASSIVE_EVALUATION = True

if RUN_MASSIVE_EVALUATION:
    print("Building fast lookup matrix for audio features...")
    # Matrix for quick search (Tensor Lookup)
    audio_matrix = np.zeros((num_items, len(audio_features)))
    df_audio_indexed = df_engineered_2.set_index("track_id")[audio_features]

    for track_id, item_idx in track_map.items():
        if track_id in df_audio_indexed.index:
            audio_matrix[item_idx] = df_audio_indexed.loc[track_id].values

    audio_tensor_lookup = torch.tensor(audio_matrix, dtype=torch.float32)

    np.random.seed(RANDOM_SEED)
    test_users_loo = test_df["user_idx"].unique()
    all_items = np.array(list(track_map.values()))

    k_values = [5, 10, 20]

    # NUEVO: Diccionario para capturar las métricas de AMBOS modelos
    metrics = {f"DL_Recall@{k}": [] for k in k_values}
    metrics.update({f"DL_NDCG@{k}": [] for k in k_values})
    metrics.update({f"SVD_Recall@{k}": [] for k in k_values})
    metrics.update({f"SVD_NDCG@{k}": [] for k in k_values})

    print(
        f"Evaluating {len(test_users_loo)} users in multi-K LOO scheme for plotting..."
    )

    model.eval()
    with torch.no_grad():
        for u in test_users_loo:
            user_positives = test_df[test_df["user_idx"] == u]
            if user_positives.empty:
                continue

            pos_item_idx = user_positives.sample(1, random_state=RANDOM_SEED)[
                "item_idx"
            ].values[0]
            user_all_interacted = df_interactions[df_interactions["user_idx"] == u][
                "item_idx"
            ].values
            valid_negatives = np.setdiff1d(all_items, user_all_interacted)

            if len(valid_negatives) < 99:
                continue
            neg_items_idx = np.random.choice(valid_negatives, size=99, replace=False)

            eval_items = np.append(neg_items_idx, pos_item_idx)
            eval_users = np.full(100, u)

            # Neural Network Prediction
            u_tensor = torch.tensor(eval_users, dtype=torch.long)
            i_tensor = torch.tensor(eval_items, dtype=torch.long)
            a_tensor = audio_tensor_lookup[i_tensor]
            dl_preds = model(u_tensor, i_tensor, a_tensor).numpy()

            # SVD prediction
            svd_user_vec = user_factors[u]
            svd_item_vecs = item_factors[:, eval_items].T
            svd_preds = np.dot(svd_item_vecs, svd_user_vec)

            true_labels = np.zeros(100)
            true_labels[-1] = 1.0

            # Metrics
            for k in k_values:
                metrics[f"DL_NDCG@{k}"].append(
                    ndcg_score([true_labels], [dl_preds], k=k)
                )
                metrics[f"SVD_NDCG@{k}"].append(
                    ndcg_score([true_labels], [svd_preds], k=k)
                )

                dl_top_k_indices = np.argsort(dl_preds)[-k:]
                metrics[f"DL_Recall@{k}"].append(1 if 99 in dl_top_k_indices else 0)

                svd_top_k_indices = np.argsort(svd_preds)[-k:]
                metrics[f"SVD_Recall@{k}"].append(1 if 99 in svd_top_k_indices else 0)

    # Save cache
    with open("cache/loo_multi_k_metrics_fixed.pkl", "wb") as f:
        pickle.dump(metrics, f)
    print(
        "Multi-K LOO metrics successfully saved to 'cache/loo_multi_k_metrics_fixed.pkl'"
    )

else:
    print("Loading pre-calculated LOO metrics from cache...")
    with open("cache/loo_multi_k_metrics_fixed.pkl", "rb") as f:
        metrics = pickle.load(f)

# Average metrics
k_values = [5, 10, 20]
avg_metrics = {key: np.mean(val) for key, val in metrics.items()}

# --- PLOT RESULTS ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(k_values))
width = 0.35

# Recall Plot: SVD vs DL
svd_recalls = [avg_metrics[f"SVD_Recall@{k}"] for k in k_values]
dl_recalls = [avg_metrics[f"DL_Recall@{k}"] for k in k_values]

axes[0].bar(x - width / 2, svd_recalls, width, label="SVD", color="#9E9E9E")
axes[0].bar(x + width / 2, dl_recalls, width, label="Two-Tower DL", color="#2196F3")
axes[0].set_ylabel("Recall Score")
axes[0].set_title("Model Comparison: Recall (Leave-One-Out)")
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"Top {k}" for k in k_values])
axes[0].legend()
axes[0].set_ylim(0, max(max(svd_recalls), max(dl_recalls)) * 1.2)

for i, (s, d) in enumerate(zip(svd_recalls, dl_recalls)):
    axes[0].text(
        i - width / 2, s + 0.01, f"{s:.3f}", ha="center", va="bottom", fontsize=9
    )
    axes[0].text(
        i + width / 2, d + 0.01, f"{d:.3f}", ha="center", va="bottom", fontsize=9
    )

# NDCG Plot: SVD vs DL
svd_ndcgs = [avg_metrics[f"SVD_NDCG@{k}"] for k in k_values]
dl_ndcgs = [avg_metrics[f"DL_NDCG@{k}"] for k in k_values]

axes[1].bar(x - width / 2, svd_ndcgs, width, label="SVD", color="#9E9E9E")
axes[1].bar(x + width / 2, dl_ndcgs, width, label="Two-Tower DL", color="#FFC107")
axes[1].set_ylabel("NDCG Score")
axes[1].set_title("Model Comparison: NDCG (Leave-One-Out)")
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"Top {k}" for k in k_values])
axes[1].legend()
axes[1].set_ylim(0, max(max(svd_ndcgs), max(dl_ndcgs)) * 1.2)

for i, (s, d) in enumerate(zip(svd_ndcgs, dl_ndcgs)):
    axes[1].text(
        i - width / 2, s + 0.01, f"{s:.3f}", ha="center", va="bottom", fontsize=9
    )
    axes[1].text(
        i + width / 2, d + 0.01, f"{d:.3f}", ha="center", va="bottom", fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
# load cache
with open("cache/dl_cache.pkl", "rb") as f:
    dl_cache = pickle.load(f)

epoch_losses = dl_cache["epoch_losses"]
epochs = list(range(1, len(epoch_losses) + 1))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 1. TRAINING LOSS CURVE ---
axes[0].plot(epochs, epoch_losses, marker="o", color="#E91E63", linewidth=2)
axes[0].set_title("Deep Learning Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary Cross-Entropy Loss")
axes[0].set_xticks(epochs)
axes[0].grid(True, linestyle="--", alpha=0.6)

# --- 2. BASELINE VS DEEP LEARNING RMSE ---
models = ["TruncatedSVD\n(Baseline)", "Two-Tower DL"]
rmse_values = [0.8879, 0.0590]

axes[1].bar(models, rmse_values, color=["#9E9E9E", "#4CAF50"], width=0.5)
axes[1].set_title("Model Convergence Comparison (RMSE)")
axes[1].set_ylabel("RMSE (Lower is better)")
axes[1].set_ylim(0, 1.0)
for i, v in enumerate(rmse_values):
    axes[1].text(i, v + 0.02, f"{v:.4f}", ha="center", va="bottom", fontweight="bold")

# --- 3. SUBGROUP ANALYSIS (COLD-START VS POWER USERS) ---
print("Computing subgroup analysis for algorithmic fairness...")

model.eval()
dl_preds = []
with torch.no_grad():
    for batch_users, batch_items, batch_audio, batch_ratings in test_loader:
        preds = model(batch_users, batch_items, batch_audio)
        dl_preds.extend(preds.numpy())

test_df["dl_pred"] = dl_preds

test_users = test_df["user_idx"].values
test_items = test_df["item_idx"].values
test_df["svd_pred"] = np.sum(
    user_factors[test_users] * item_factors[:, test_items].T, axis=1
)

user_counts = train_df.groupby("user_idx").size()
test_df["train_interactions"] = test_df["user_idx"].map(user_counts).fillna(0)

cold_start_mask = test_df["train_interactions"] <= 10
power_user_mask = test_df["train_interactions"] > 10

svd_rmse_cold = np.sqrt(
    mean_squared_error(
        test_df[cold_start_mask]["rating"], test_df[cold_start_mask]["svd_pred"]
    )
)
svd_rmse_power = np.sqrt(
    mean_squared_error(
        test_df[power_user_mask]["rating"], test_df[power_user_mask]["svd_pred"]
    )
)

dl_rmse_cold = np.sqrt(
    mean_squared_error(
        test_df[cold_start_mask]["rating"], test_df[cold_start_mask]["dl_pred"]
    )
)
dl_rmse_power = np.sqrt(
    mean_squared_error(
        test_df[power_user_mask]["rating"], test_df[power_user_mask]["dl_pred"]
    )
)

labels = ["Cold-Start (<=10 tracks)", "Power Users (>10 tracks)"]
svd_cohorts = [svd_rmse_cold, svd_rmse_power]
dl_cohorts = [dl_rmse_cold, dl_rmse_power]

x = np.arange(len(labels))
width = 0.35

axes[2].bar(x - width / 2, svd_cohorts, width, label="SVD", color="#9E9E9E")
axes[2].bar(x + width / 2, dl_cohorts, width, label="Two-Tower DL", color="#4CAF50")

axes[2].set_title("Subgroup Fairness (RMSE by Playlist Size)")
axes[2].set_ylabel("RMSE")
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels)
axes[2].legend()
axes[2].set_ylim(0, 1.0)

for i, v in enumerate(svd_cohorts):
    axes[2].text(
        i - width / 2, v + 0.02, f"{v:.3f}", ha="center", va="bottom", fontsize=9
    )
for i, v in enumerate(dl_cohorts):
    axes[2].text(
        i + width / 2, v + 0.02, f"{v:.3f}", ha="center", va="bottom", fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
print("Building and saving fast lookup matrix for API...")
audio_matrix = np.zeros((num_items, len(audio_features)))
df_audio_indexed = df_engineered_2.set_index("track_id")[audio_features]

for track_id, item_idx in track_map.items():
    if track_id in df_audio_indexed.index:
        audio_matrix[item_idx] = df_audio_indexed.loc[track_id].values

audio_tensor_lookup = torch.tensor(audio_matrix, dtype=torch.float32)
torch.save(audio_tensor_lookup, "cache/audio_tensors.pt")
print("Tensor catalog exported to cache/audio_tensors.pt")